# EDUAI · Chatterbox V3 · trabajador Kaggle

Este notebook sirve a **tu perfil de voz propio** mediante una sesión GPU gratuita y temporal. No es un servidor permanente.

1. Settings → Accelerator → GPU T4 x2, activa Internet y guarda el notebook como privado.
2. Ejecuta las celdas de instalación y de pesos. La primera ejecución puede tomar varios minutos.
3. Ve a **https://eduai.innova-space-edu.cl/audio-lab/voices** y desbloquea Mis voces con MFA.
4. En la sección **Generar audio desde EDUAI**, selecciona Kaggle y pulsa **Crear código de conexión**.
5. Ejecuta la última celda; pega el código **en la entrada privada** (no en el código de la celda).
6. Mantén esa celda en ejecución mientras escribes y generas audios desde EDUAI. Al cerrar Kaggle/Colab, tendrás que volver a iniciarla con un código nuevo.

Seguridad: Solo tu voz y consentimiento; no incluyas claves de Supabase, datos de terceros ni código de conexión guardado en el notebook. Los resultados WAV quedan en la biblioteca privada de EDUAI. La marca de agua Perth permanece activada.

In [ ]:
# Notebook EDUAI: instalación aislada y compatible con Chatterbox V3
import os, sys, subprocess, shutil
from pathlib import Path

if not shutil.which("ffmpeg"):
    subprocess.run(["apt-get","update","-qq"],check=True)
    subprocess.run(["apt-get","install","-y","-qq","ffmpeg"],check=True)
import torch
assert torch.cuda.is_available(), "Activa GPU T4 en la configuración del notebook"
print("GPU:",torch.cuda.get_device_name(0))

subprocess.run([sys.executable,"-m","pip","install","-q","uv"],check=True)
UV=[sys.executable,"-m","uv"]
BASE=Path("/kaggle/working" if Path("/kaggle/working").exists() else "/content")
ENV=BASE/"eduai_voice_env"
PY=str(ENV/"bin"/"python")
subprocess.run(UV+["venv","--python","3.11",str(ENV)],check=True)
def pip_uv(*args):
    subprocess.run(UV+["pip","install","--python",PY,*args],check=True)
pip_uv("torch==2.6.0","torchvision==0.21.0","torchaudio==2.6.0",
    "--index-url","https://download.pytorch.org/whl/cu126")
pip_uv("chatterbox-tts==0.1.7","modelscope>=1.20,<3","requests","setuptools<81")
pip_uv("--reinstall","--no-deps",
    "git+https://github.com/resemble-ai/chatterbox.git@5de7a54aa4e5e2baadb0182dde554908b48b85c2")
test=subprocess.run([PY,"-c",(
    "import torch, torchaudio, torchvision, perth, inspect;"
    "from chatterbox.mtl_tts import ChatterboxMultilingualTTS;"
    "assert torch.cuda.is_available();"
    "assert perth.PerthImplicitWatermarker is not None;"
    "assert 't3_model' in inspect.signature(ChatterboxMultilingualTTS.from_local).parameters;"
    "print('EDUAI CHATTERBOX V3 Y MARCA PERTH LISTOS')")],check=True)
print("Entorno independiente:",PY)


In [ ]:
# Descargar pesos una vez por sesión desde ModelScope (no desde HF).
import subprocess, os
from pathlib import Path
BASE=Path("/kaggle/working" if Path("/kaggle/working").exists() else "/content")
PY=str(BASE/"eduai_voice_env"/"bin"/"python")
MODEL_DIR=BASE/"eduai_model_v3"
MODEL_DIR.mkdir(parents=True,exist_ok=True)
NEEDED=["ve.pt","s3gen.pt","t3_mtl23ls_v3.safetensors",
        "grapheme_mtl_merged_expanded_v1.json"]
missing=[f for f in NEEDED if not (MODEL_DIR/f).exists()]
if missing:
    subprocess.run([str(Path(PY).parent/"modelscope"),"download",
       "--model","ResembleAI/chatterbox",*missing,
       "--local_dir",str(MODEL_DIR)],check=True)
assert all((MODEL_DIR/f).stat().st_size>2000 for f in NEEDED)
print("Pesos Chatterbox V3 listos en",MODEL_DIR)


In [ ]:
WORKER_CODE = "import os, io, re, sys, time, wave, json, tempfile, subprocess\nfrom pathlib import Path\nfrom urllib.parse import urlparse\nimport requests, torch, torchaudio\nfrom chatterbox.mtl_tts import ChatterboxMultilingualTTS\n\nPAIR = os.environ.pop(\"EDUAI_PAIR_CODE\", \"\")\nif not re.fullmatch(r\"[a-f0-9-]{36}\\.[a-f0-9]{64}\", PAIR):\n    raise RuntimeError(\"El código de vinculación no tiene el formato esperado\")\nsession_id, token = PAIR.split(\".\", 1)\nendpoint = \"https://eduai.innova-space-edu.cl/api/agents/audio/voices/profiles?mode=compute\"\nmodel_dir = Path(os.environ[\"EDUAI_MODEL_DIR\"])\nos.environ[\"HF_HUB_OFFLINE\"] = \"1\"\nos.environ[\"TRANSFORMERS_OFFLINE\"] = \"1\"\nif not torch.cuda.is_available():\n    raise RuntimeError(\"GPU CUDA desactivada: habilita la T4 en Settings\")\nprint(\"Cargando Chatterbox V3 en\", torch.cuda.get_device_name(0), flush=True)\nmodel = ChatterboxMultilingualTTS.from_local(ckpt_dir=model_dir,device=\"cuda\",t3_model=\"v3\")\nprint(\"Motor listo. Esperando solicitudes desde EDUAI. Detén la celda para desconectar.\",flush=True)\n\ndef call(operation, **kwargs):\n    try:\n        response = requests.post(endpoint, json={\n            \"operation\":operation,\"session_id\":session_id,\"token\":token,**kwargs\n        }, timeout=40,headers={\"Cache-Control\":\"no-store\"})\n        if response.status_code!=200:\n            raise RuntimeError(\"EDUAI respondió HTTP \"+str(response.status_code))\n        return response.json()\n    except requests.RequestException:\n        raise RuntimeError(\"No se pudo conectar a EDUAI; revisa Internet\") from None\n\ndef permitted_signed_url(url, upload=False):\n    parsed = urlparse(url)\n    if parsed.scheme!=\"https\" or not (parsed.hostname or \"\").endswith(\".supabase.co\"):\n        raise ValueError(\"URL de almacenamiento no autorizada\")\n    root = \"/storage/v1/object/upload/sign/\" if upload else \"/storage/v1/object/sign/\"\n    if root not in parsed.path or not parsed.query:\n        raise ValueError(\"Firma temporal de archivo inválida\")\n    return url\n\nuntil=time.monotonic()+115*60\nwhile time.monotonic()<until:\n    try:\n        response=call(\"poll\")\n        if not response.get(\"ok\"):\n            print(\"La vinculación caducó o se desconectó; finalizando.\",flush=True)\n            break\n        job=response.get(\"job\")\n        if not job:\n            time.sleep(7)\n            continue\n        print(\"Generando audio de\",len(job[\"text\"]),\"caracteres...\",flush=True)\n        try:\n            source_url=permitted_signed_url(job[\"reference_url\"])\n            src=requests.get(source_url,timeout=60)\n            if src.status_code!=200 or len(src.content)>12*1024*1024 or len(src.content)<1500:\n                raise RuntimeError(\"La muestra vocal no está accesible o es demasiado grande\")\n            with tempfile.TemporaryDirectory(prefix=\"eduai-voice-\") as tmp:\n                original=Path(tmp)/\"source.bin\"\n                reference=Path(tmp)/\"reference.wav\"\n                output=Path(tmp)/\"output.wav\"\n                original.write_bytes(src.content)\n                subprocess.run([\"ffmpeg\",\"-hide_banner\",\"-loglevel\",\"error\",\"-y\",\n                     \"-i\",str(original),\"-ac\",\"1\",\"-ar\",\"24000\",\"-acodec\",\"pcm_s16le\",str(reference)],\n                     check=True,timeout=60)\n                with wave.open(str(reference),\"rb\") as sound:\n                    seconds=sound.getnframes()/sound.getframerate()\n                if seconds<8 or seconds>35:\n                    raise RuntimeError(\"La referencia debe durar aproximadamente de 10 a 30 segundos\")\n                with torch.inference_mode():\n                    audio=model.generate(job[\"text\"],language_id=\"es\",audio_prompt_path=str(reference))\n                torchaudio.save(str(output),audio.detach().cpu(),model.sr,format=\"wav\",\n                    encoding=\"PCM_S\",bits_per_sample=16)\n                wav=output.read_bytes()\n                if len(wav)>4*1024*1024:\n                    raise RuntimeError(\"Audio mayor a 4 MB; divide el texto en frases\")\n                url=permitted_signed_url(job[\"upload_url\"],upload=True)\n                saved=requests.put(url,data=wav,headers={\"Content-Type\":\"audio/wav\",\"x-upsert\":\"false\"},\n                    timeout=120)\n                if saved.status_code not in (200,201):\n                    raise RuntimeError(\"Supabase rechazó el WAV (HTTP \"+str(saved.status_code)+\")\")\n            for attempt in range(4):\n                done=call(\"finish\",job_id=job[\"id\"])\n                if done.get(\"ok\"):\n                    print(\"Audio guardado en EDUAI. Ya puedes reproducirlo en Mis voces.\",flush=True)\n                    break\n                time.sleep(1)\n            else:\n                raise RuntimeError(\"EDUAI no confirmó el archivo guardado\")\n        except Exception as exc:\n            # Never print a signed URL, access token, or private audio bytes.\n            print(\"Falló el trabajo:\",type(exc).__name__,flush=True)\n            try:\n                call(\"finish\",job_id=job[\"id\"],error=\"Fallo del motor GPU: \"+type(exc).__name__)\n            except Exception:\n                pass\n        torch.cuda.empty_cache()\n    except KeyboardInterrupt:\n        print(\"Trabajador detenido por el usuario.\",flush=True)\n        break\n    except Exception as exc:\n        print(\"Conexión interrumpida:\",type(exc).__name__,flush=True)\n        time.sleep(10)\nprint(\"Sesión GPU terminada.\",flush=True)\n"
print('Código de trabajador EDUAI preparado. No se ejecuta hasta vincular la GPU.')

In [ ]:
# Vincular con EDUAI. El código no se guarda en el notebook ni se muestra.
import getpass, os, subprocess
from pathlib import Path
BASE=Path("/kaggle/working" if Path("/kaggle/working").exists() else "/content")
PY=str(BASE/"eduai_voice_env"/"bin"/"python")
MODEL_DIR=BASE/"eduai_model_v3"
PAIR=getpass.getpass("Pega aquí el código temporal de EDUAI Mis voces: ").strip()
if not PAIR:
    raise ValueError("Obtén un código en EDUAI: Mis voces → Crear código de conexión")
SCRIPT=BASE/"eduai_gpu_worker.py"
SCRIPT.write_text(WORKER_CODE,encoding="utf-8")
environment=os.environ.copy()
environment["EDUAI_PAIR_CODE"]=PAIR
environment["EDUAI_MODEL_DIR"]=str(MODEL_DIR)
environment["HF_HUB_OFFLINE"]="1"
environment["TRANSFORMERS_OFFLINE"]="1"
del PAIR
print("Iniciando trabajador de voz. Vuelve a EDUAI y pulsa Generar audio.")
try:
    subprocess.run([PY,str(SCRIPT)],env=environment,check=True)
finally:
    environment.pop("EDUAI_PAIR_CODE",None)
